# Dataset Exploration — Candidate Benchmarks for Prompt Piper

This notebook explores five lm-eval tasks selected as candidates for evaluating the prompt compressor:

| # | Task name | Domain | Few-shot | Eval type |
|---|-----------|--------|----------|-----------|
| 1 | `mbpp_instruct` | Python code generation | 3-shot | pass@1 |
| 2 | `mmlu_pro_computer_science` | CS multiple-choice (CoT) | 5-shot | accuracy |
| 3 | `gsm8k_cot` | Math word problems (CoT) | 8-shot | exact match |
| 4 | `bbh_cot_fewshot_logical_deduction_seven_objects` | Logical deduction (CoT) | 3-shot | exact match |
| 5 | `leaderboard_musr_murder_mysteries` | Long-narrative reasoning | 0-shot | accuracy |

For each task we show:
- The **system instruction / task description** and prompt template
- The **few-shot block** (rendered as text)
- **3 data samples** with fully rendered prompts and token-count breakdown
- A **statistics summary** (prompt length distribution)

In [4]:
import os
# Required by the HuggingFace code_eval metric used by MBPP.
# We only load the task for data inspection — no model code is executed.
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

import sys
sys.path.insert(0, "../src")

import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 120)

from prompt_piper.backends.lm_eval import LMEvalDataset
from prompt_piper.optimization.tokenizer import WhitespaceTokenizer

tokenizer = WhitespaceTokenizer()

# ── helpers ──────────────────────────────────────────────────────────────────

WRAP = 100   # characters per line for pretty-printing

def wrap_print(text: str, indent: int = 2) -> None:
    """Print text with hard line-wrap and optional indent."""
    prefix = " " * indent
    for line in text.splitlines():
        if len(line) <= WRAP:
            print(prefix + line)
        else:
            for chunk in textwrap.wrap(line, width=WRAP):
                print(prefix + chunk)

def section(title: str) -> None:
    bar = "═" * (WRAP + 2)
    print(f"\n{bar}")
    print(f"  {title}")
    print(bar)

def subsection(title: str) -> None:
    print(f"\n── {title} {'─' * (WRAP - len(title) - 4)}")

def token_stats(texts: list[str]) -> dict:
    counts = [tokenizer.count(t) for t in texts]
    return {
        "n": len(counts),
        "mean": round(float(np.mean(counts)), 1),
        "median": round(float(np.median(counts)), 1),
        "min": int(np.min(counts)),
        "max": int(np.max(counts)),
        "p95": round(float(np.percentile(counts, 95)), 1),
    }

print("Setup complete.")

Setup complete.


---
## 1. MBPP Instruct — Python Code Generation

**Task:** Given a natural-language task description and three test cases, generate a Python function that passes all tests.  
**Metric:** pass@1 (functional correctness via code execution).  
**Few-shot:** 3 shots, each showing a task + tests + solution.  
**Compression opportunity:** The instruction preamble, test-case formatting, and few-shot solutions can all be compressed.

**Config source:** `.venv/lib/python3.12/site-packages/lm_eval/tasks/mbpp/mbpp_instruct.yaml`

In [5]:
# ── Prompt template (from YAML) ───────────────────────────────────────────────
MBPP_TEMPLATE = """\
You are an expert Python programmer, and here is your task:
{{text}}
Your code should pass these tests:
{{test_list[0]}}
{{test_list[1]}}
{{test_list[2]}}"""

section("MBPP Instruct — Prompt Template")
print()
wrap_print(MBPP_TEMPLATE)


══════════════════════════════════════════════════════════════════════════════════════════════════════
  MBPP Instruct — Prompt Template
══════════════════════════════════════════════════════════════════════════════════════════════════════

  You are an expert Python programmer, and here is your task:
  {{text}}
  Your code should pass these tests:
  {{test_list[0]}}
  {{test_list[1]}}
  {{test_list[2]}}


In [6]:
# ── Few-shot examples (from mbpp/utils.py) ───────────────────────────────────
# lm-eval hard-codes the first 3 training samples as fewshot_samples.
# We reproduce them here for inspection.

from lm_eval.tasks.mbpp.utils import list_fewshot_samples

shots = list_fewshot_samples()

section("MBPP Instruct — Few-Shot Examples (3 shots)")
for i, shot in enumerate(shots):
    subsection(f"Shot {i+1}")
    prompt_part = (
        f"You are an expert Python programmer, and here is your task:\n"
        f"{shot['text']}\n"
        f"Your code should pass these tests:\n"
        + "\n".join(shot["test_list"])
    )
    wrap_print("[INSTRUCTION + TESTS]")
    wrap_print(prompt_part)
    print()
    wrap_print("[SOLUTION]")
    wrap_print(shot["code"])
    print(f"\n  → prompt tokens: {tokenizer.count(prompt_part)} | "
          f"solution tokens: {tokenizer.count(shot['code'])}")


══════════════════════════════════════════════════════════════════════════════════════════════════════
  MBPP Instruct — Few-Shot Examples (3 shots)
══════════════════════════════════════════════════════════════════════════════════════════════════════

── Shot 1 ──────────────────────────────────────────────────────────────────────────────────────────
  [INSTRUCTION + TESTS]
  You are an expert Python programmer, and here is your task:
  Write a function to find the similar elements from the given two tuple lists.
  Your code should pass these tests:
  assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)
  assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)
  assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)

  [SOLUTION]
  def similar_elements(test_tup1, test_tup2):
    res = tuple(set(test_tup1) & set(test_tup2))
    return (res) 

  → prompt tokens: 64 | solution tokens: 10

── Shot 2 ───────────────────────────────────────────────────────────

In [4]:
# ── Load dataset and show 3 rendered samples ──────────────────────────────────
print("Loading mbpp_instruct (test split, 3-shot)…")
mbpp_ds = LMEvalDataset.from_task_name(
    "mbpp_instruct",
    split="test",
    num_fewshot=3,
    limit=50,
    template_field="prompt_0shot",
).as_list()
print(f"Loaded {len(mbpp_ds)} samples.")

section("MBPP Instruct — 3 Rendered Samples (3-shot)")
for i, sample in enumerate(mbpp_ds[:3]):
    subsection(f"Sample {i+1}")
    full   = sample["prompt"]
    zshot  = sample.get("prompt_0shot", "")
    shots_block = sample.get("_shots", "")
    tgt    = sample["target"]

    print(f"  Full prompt tokens   : {tokenizer.count(full)}")
    print(f"  0-shot prompt tokens : {tokenizer.count(zshot)}")
    print(f"  Few-shot block tokens: {tokenizer.count(shots_block)}")
    print()
    wrap_print("[FEW-SHOT BLOCK (first 300 chars)]")
    wrap_print(shots_block[:300] + ("…" if len(shots_block) > 300 else ""))
    print()
    wrap_print("[0-SHOT PROMPT]")
    wrap_print(zshot)
    print()
    wrap_print(f"[TARGET] {tgt[:200]}")

Loading mbpp_instruct (test split, 3-shot)…


Loaded 50 samples.

══════════════════════════════════════════════════════════════════════════════════════════════════════
  MBPP Instruct — 3 Rendered Samples (3-shot)
══════════════════════════════════════════════════════════════════════════════════════════════════════

── Sample 1 ────────────────────────────────────────────────────────────────────────────────────────
  Full prompt tokens   : 274
  0-shot prompt tokens : 46
  Few-shot block tokens: 228

  [FEW-SHOT BLOCK (first 300 chars)]
  You are an expert Python programmer, and here is your task:
  Write a function to find the similar elements from the given two tuple lists.
  Your code should pass these tests:
  assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)
  assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)
  asse…

  [0-SHOT PROMPT]
  You are an expert Python programmer, and here is your task:
  Write a python function to remove first and last occurrence of a given character from the string.
  Your 

In [5]:
# ── Token-count distribution ──────────────────────────────────────────────────
mbpp_full_stats  = token_stats([s["prompt"] for s in mbpp_ds])
mbpp_zshot_stats = token_stats([s.get("prompt_0shot", "") for s in mbpp_ds])
mbpp_shots_stats = token_stats([s.get("_shots", "") for s in mbpp_ds])

df_mbpp = pd.DataFrame(
    [mbpp_full_stats, mbpp_zshot_stats, mbpp_shots_stats],
    index=["full prompt", "0-shot prompt", "few-shot block"]
)
print("MBPP Instruct — Token Statistics")
print(df_mbpp.to_string())

MBPP Instruct — Token Statistics
                 n   mean  median  min  max    p95
full prompt     50  277.8   274.0  260  351  307.6
0-shot prompt   50   49.8    46.0   32  123   79.6
few-shot block  50  228.0   228.0  228  228  228.0


---
## 2. MMLU-Pro Computer Science — Multiple-Choice CoT

**Task:** 10-option multiple-choice questions covering CS topics (algorithms, data structures, OS, networking, programming).  
**Metric:** Exact match on the selected letter (A–J).  
**Few-shot:** 5 shots, each with a full chain-of-thought solution.  
**Compression opportunity:** The CoT rationale in each shot is long (~100–200 tokens per shot) and partially compressible. The description preamble and "Think step by step" instruction can also be shortened.

**Config source:** `.venv/lib/python3.12/site-packages/lm_eval/tasks/mmlu_pro/mmlu_pro_computer_science.yaml`

In [ ]:
MMLU_DESCRIPTION = (
    'The following are multiple choice questions (with answers) about computer science. '
    'Think step by step and then finish your answer with "the answer is (X)" '
    'where X is the correct letter choice.'
)

MMLU_TEMPLATE = """\
Question:
{{question}}
Options:
A. {{options[0]}}
B. {{options[1]}}
...
J. {{options[9]}}
Answer: Let's think step by step."""

section("MMLU-Pro CS — Task Description & Prompt Template")
subsection("Description (prepended to every prompt)")
wrap_print(MMLU_DESCRIPTION)
subsection("Prompt template")
wrap_print(MMLU_TEMPLATE)

In [ ]:
print("Loading mmlu_pro_computer_science (test split, 5-shot)…")
mmlu_ds = LMEvalDataset.from_task_name(
    "mmlu_pro_computer_science",
    split="test",
    num_fewshot=5,
    limit=50,
    template_field="prompt_0shot",
).as_list()
print(f"Loaded {len(mmlu_ds)} samples.")

section("MMLU-Pro CS — 3 Rendered Samples (5-shot)")
for i, sample in enumerate(mmlu_ds[:3]):
    subsection(f"Sample {i+1}")
    full  = sample["prompt"]
    zshot = sample.get("prompt_0shot", "")
    shots_block = sample.get("_shots", "")
    tgt   = sample["target"]

    print(f"  Full prompt tokens   : {tokenizer.count(full)}")
    print(f"  0-shot prompt tokens : {tokenizer.count(zshot)}")
    print(f"  Few-shot block tokens: {tokenizer.count(shots_block)}")
    print()
    wrap_print("[FEW-SHOT BLOCK (first 400 chars)]")
    wrap_print(shots_block[:400] + ("…" if len(shots_block) > 400 else ""))
    print()
    wrap_print("[0-SHOT PROMPT]")
    wrap_print(zshot)
    print()
    wrap_print(f"[TARGET] {tgt}")

In [ ]:
mmlu_full_stats  = token_stats([s["prompt"] for s in mmlu_ds])
mmlu_zshot_stats = token_stats([s.get("prompt_0shot", "") for s in mmlu_ds])
mmlu_shots_stats = token_stats([s.get("_shots", "") for s in mmlu_ds])

df_mmlu = pd.DataFrame(
    [mmlu_full_stats, mmlu_zshot_stats, mmlu_shots_stats],
    index=["full prompt", "0-shot prompt", "few-shot block"]
)
print("MMLU-Pro CS — Token Statistics")
print(df_mmlu.to_string())

---
## 3. GSM8K CoT — Grade-School Math Word Problems

**Task:** Multi-step arithmetic word problems requiring chain-of-thought reasoning.  
**Metric:** Exact match on the final numeric answer.  
**Few-shot:** 8 shots, each containing a full reasoning chain.  
**Compression opportunity:** The 8 shot reasoning chains dominate prompt length (~400–500 tokens). The per-shot rationale contains many words that could be condensed without losing the arithmetic steps.

**Config source:** `.venv/lib/python3.12/site-packages/lm_eval/tasks/gsm8k/gsm8k-cot.yaml`

In [ ]:
GSM_TEMPLATE = """\
Given the following problem, reason and give a final answer to the problem.
Problem: {{question}}
Your response should end with "The final answer is [answer]" where [answer] is the response to the problem."""

# Hard-coded few-shot samples from the gsm8k-cot.yaml
GSM_SHOTS = [
    {
        "question": "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?",
        "answer": "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6",
    },
    {
        "question": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?",
        "answer": "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The final answer is 5",
    },
    {
        "question": "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        "answer": "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The final answer is 39",
    },
]

section("GSM8K CoT — Prompt Template")
wrap_print(GSM_TEMPLATE)

section("GSM8K CoT — First 3 of 8 Few-Shot Examples")
for i, shot in enumerate(GSM_SHOTS):
    subsection(f"Shot {i+1}")
    q = f"Given the following problem, reason and give a final answer to the problem.\nProblem: {shot['question']}\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem."
    wrap_print("[QUESTION]")
    wrap_print(q)
    print()
    wrap_print("[REASONING CHAIN]")
    wrap_print(shot["answer"])
    print(f"\n  → question tokens: {tokenizer.count(q)} | chain tokens: {tokenizer.count(shot['answer'])}")

In [ ]:
print("Loading gsm8k_cot (test split, 8-shot)…")
gsm_ds = LMEvalDataset.from_task_name(
    "gsm8k_cot",
    split="test",
    num_fewshot=8,
    limit=50,
    template_field="prompt_0shot",
).as_list()
print(f"Loaded {len(gsm_ds)} samples.")

section("GSM8K CoT — 3 Rendered Samples (8-shot)")
for i, sample in enumerate(gsm_ds[:3]):
    subsection(f"Sample {i+1}")
    full  = sample["prompt"]
    zshot = sample.get("prompt_0shot", "")
    shots_block = sample.get("_shots", "")
    tgt   = sample["target"]

    print(f"  Full prompt tokens   : {tokenizer.count(full)}")
    print(f"  0-shot prompt tokens : {tokenizer.count(zshot)}")
    print(f"  Few-shot block tokens: {tokenizer.count(shots_block)}")
    print()
    wrap_print("[FEW-SHOT BLOCK (first 500 chars)]")
    wrap_print(shots_block[:500] + ("…" if len(shots_block) > 500 else ""))
    print()
    wrap_print("[0-SHOT PROMPT]")
    wrap_print(zshot)
    print()
    wrap_print(f"[TARGET] {tgt[:200]}")

In [ ]:
gsm_full_stats  = token_stats([s["prompt"] for s in gsm_ds])
gsm_zshot_stats = token_stats([s.get("prompt_0shot", "") for s in gsm_ds])
gsm_shots_stats = token_stats([s.get("_shots", "") for s in gsm_ds])

df_gsm = pd.DataFrame(
    [gsm_full_stats, gsm_zshot_stats, gsm_shots_stats],
    index=["full prompt", "0-shot prompt", "few-shot block"]
)
print("GSM8K CoT — Token Statistics")
print(df_gsm.to_string())

---
## 4. BBH CoT — Logical Deduction (Seven Objects)

**Task:** Given a set of logical constraints about the relative ordering of 7 objects, determine a specific object's position.  
**Metric:** Exact match on the selected option letter.  
**Few-shot:** 3 shots, each with a multi-paragraph constraint description and a full chain-of-thought answer.  
**Compression opportunity:** The constraint narratives in each shot are long and highly structured — repeated phrases like "is to the left of" / "is immediately to the right" are prime targets for syntactic compression.

**Config source:** `.venv/lib/python3.12/site-packages/lm_eval/tasks/bbh/cot_fewshot/logical_deduction_seven_objects.yaml`

In [ ]:
BBH_DESCRIPTION = "A logical deduction task which requires deducing the order of a sequence of objects."
BBH_TEMPLATE = "Q: {{input}}\nA: Let's think step by step."

section("BBH Logical Deduction (7 objects) — Task Description & Prompt Template")
subsection("Description")
wrap_print(BBH_DESCRIPTION)
subsection("Prompt template")
wrap_print(BBH_TEMPLATE)

In [ ]:
print("Loading bbh_cot_fewshot_logical_deduction_seven_objects (test split, 3-shot)…")
bbh_ds = LMEvalDataset.from_task_name(
    "bbh_cot_fewshot_logical_deduction_seven_objects",
    split="test",
    num_fewshot=3,
    limit=50,
    template_field="prompt_0shot",
).as_list()
print(f"Loaded {len(bbh_ds)} samples.")

section("BBH Logical Deduction — 3 Rendered Samples (3-shot)")
for i, sample in enumerate(bbh_ds[:3]):
    subsection(f"Sample {i+1}")
    full  = sample["prompt"]
    zshot = sample.get("prompt_0shot", "")
    shots_block = sample.get("_shots", "")
    tgt   = sample["target"]

    print(f"  Full prompt tokens   : {tokenizer.count(full)}")
    print(f"  0-shot prompt tokens : {tokenizer.count(zshot)}")
    print(f"  Few-shot block tokens: {tokenizer.count(shots_block)}")
    print()
    wrap_print("[FEW-SHOT BLOCK (first 600 chars)]")
    wrap_print(shots_block[:600] + ("…" if len(shots_block) > 600 else ""))
    print()
    wrap_print("[0-SHOT PROMPT]")
    wrap_print(zshot)
    print()
    wrap_print(f"[TARGET] {tgt}")

In [ ]:
bbh_full_stats  = token_stats([s["prompt"] for s in bbh_ds])
bbh_zshot_stats = token_stats([s.get("prompt_0shot", "") for s in bbh_ds])
bbh_shots_stats = token_stats([s.get("_shots", "") for s in bbh_ds])

df_bbh = pd.DataFrame(
    [bbh_full_stats, bbh_zshot_stats, bbh_shots_stats],
    index=["full prompt", "0-shot prompt", "few-shot block"]
)
print("BBH Logical Deduction — Token Statistics")
print(df_bbh.to_string())

---
## 5. MuSR Murder Mysteries — Long-Narrative Reasoning

**Task:** Read a multi-paragraph murder mystery story, then answer a multiple-choice question about who committed the crime and why.  
**Metric:** Normalised accuracy (`acc_norm`) on the selected option.  
**Few-shot:** None (0-shot by design — each narrative is unique).  
**Compression opportunity:** The narrative field can be 500–1,000+ words of story text. Structural (deduplication, section pruning) and semantic (context compression) operators have the highest potential here.

**Config source:** `.venv/lib/python3.12/site-packages/lm_eval/tasks/leaderboard/musr/`

In [ ]:
MUSR_TEMPLATE = """\
{narrative}

{question}

1 - {choice_0}
2 - {choice_1}
...
Answer:"""

section("MuSR Murder Mysteries — Prompt Template")
wrap_print(MUSR_TEMPLATE)

In [ ]:
print("Loading leaderboard_musr_murder_mysteries (test split, 0-shot)…")
musr_ds = LMEvalDataset.from_task_name(
    "leaderboard_musr_murder_mysteries",
    split="test",
    num_fewshot=0,
    limit=50,
    template_field="prompt_0shot",
).as_list()
print(f"Loaded {len(musr_ds)} samples.")

section("MuSR Murder Mysteries — 3 Rendered Samples (0-shot)")
for i, sample in enumerate(musr_ds[:3]):
    subsection(f"Sample {i+1}")
    full = sample["prompt"]
    tgt  = sample["target"]

    print(f"  Full prompt tokens: {tokenizer.count(full)}")
    print()
    # Split narrative from question
    parts = full.rsplit("\n\n", 2)
    narrative = parts[0] if len(parts) >= 2 else full
    remainder = "\n\n".join(parts[1:]) if len(parts) >= 2 else ""

    wrap_print("[NARRATIVE (first 600 chars)]")
    wrap_print(narrative[:600] + ("…" if len(narrative) > 600 else ""))
    print(f"  (narrative tokens: {tokenizer.count(narrative)})")
    print()
    wrap_print("[QUESTION + CHOICES]")
    wrap_print(remainder[:400] + ("…" if len(remainder) > 400 else ""))
    print()
    wrap_print(f"[TARGET] {tgt}")

In [ ]:
musr_full_stats = token_stats([s["prompt"] for s in musr_ds])

df_musr = pd.DataFrame(
    [musr_full_stats],
    index=["full prompt (= narrative + question)"]
)
print("MuSR Murder Mysteries — Token Statistics")
print(df_musr.to_string())

---
## Cross-Task Comparison

Summary statistics and visualisations comparing all five tasks.

---
## 6. CodeXGlue code2text — Code Summarisation (Contrast Case)

**Task:** Given a tokenised Python function body, generate a one-line docstring.  
**Metric:** Smoothed BLEU-4 against the reference docstring.  
**Few-shot:** None (0-shot only — the task has no few-shot config).  
**Prompt structure:** The *entire* prompt is the code, space-joined: `def foo ( x ) : return x * 2`.

This task is included here as a **contrast case** to illustrate why `code2text` is a poor fit
for the prompt compressor, despite being a genuine SE benchmark.

**Config source:** `.venv/lib/python3.12/site-packages/lm_eval/tasks/code_x_glue/code-text/python.yaml`

In [2]:
CODE2TEXT_TEMPLATE = """\
# doc_to_text from utils.py — no instruction whatsoever:
inputs = " ".join(doc["code_tokens"]).replace("\\n", " ")
inputs = " ".join(inputs.strip().split())
# → the entire prompt IS the code, space-tokenised"""

section("CodeXGlue code2text — Prompt Template")
subsection("Template (from utils.py)")
wrap_print(CODE2TEXT_TEMPLATE)
print()
wrap_print("There is no system instruction, no role, no task description.")
wrap_print("There is no fewshot_config in the YAML — only 0-shot is supported.")


══════════════════════════════════════════════════════════════════════════════════════════════════════
  CodeXGlue code2text — Prompt Template
══════════════════════════════════════════════════════════════════════════════════════════════════════

── Template (from utils.py) ────────────────────────────────────────────────────────────────────────
  # doc_to_text from utils.py — no instruction whatsoever:
  inputs = " ".join(doc["code_tokens"]).replace("\n", " ")
  inputs = " ".join(inputs.strip().split())
  # → the entire prompt IS the code, space-tokenised

  There is no system instruction, no role, no task description.
  There is no fewshot_config in the YAML — only 0-shot is supported.


In [3]:
print("Loading code2text_python (test split, 0-shot)…")
code2text_ds = LMEvalDataset.from_task_name(
    "code2text_python",
    split="test",
    num_fewshot=0,
    limit=50,
).as_list()
print(f"Loaded {len(code2text_ds)} samples.")

section("CodeXGlue code2text — 3 Rendered Samples (0-shot)")
for i, sample in enumerate(code2text_ds[:3]):
    subsection(f"Sample {i+1}")
    prompt = sample["prompt"]
    tgt    = sample["target"]
    print(f"  Prompt tokens: {tokenizer.count(prompt)}")
    print()
    wrap_print("[PROMPT — tokenised code, space-joined]")
    wrap_print(prompt[:400] + ("…" if len(prompt) > 400 else ""))
    print()
    wrap_print(f"[TARGET — reference docstring] {tgt}")

Loading code2text_python (test split, 0-shot)…


/home/vincenzoscotti/PycharmProjects/prompt_piper/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 50 samples.

══════════════════════════════════════════════════════════════════════════════════════════════════════
  CodeXGlue code2text — 3 Rendered Samples (0-shot)
══════════════════════════════════════════════════════════════════════════════════════════════════════

── Sample 1 ────────────────────────────────────────────────────────────────────────────────────────
  Prompt tokens: 52

  [PROMPT — tokenised code, space-joined]
  def sina_xml_to_url_list ( xml_data ) : rawurl = [ ] dom = parseString ( xml_data ) for node in dom
  . getElementsByTagName ( 'durl' ) : url = node . getElementsByTagName ( 'url' ) [ 0 ] rawurl .
  append ( url . childNodes [ 0 ] . data ) return rawurl

  [TARGET — reference docstring] str - > list Convert XML to URL List . From Biligrab .

── Sample 2 ────────────────────────────────────────────────────────────────────────────────────────
  Prompt tokens: 150

  [PROMPT — tokenised code, space-joined]
  def dailymotion_download ( url , output_dir 

In [ ]:
# ── Show what each operator does to a code prompt ────────────────────────────
from prompt_piper.transformations.lexical import (
    RemoveExtraWhitespace, RemoveStopwords, RemovePolitenessMarkers,
    ContractionSubstitution,
)
from prompt_piper.transformations.structural import (
    DeduplicateInstructions, RemovePromptHeader,
)
from prompt_piper.transformations.syntactic import SentenceMerge

sample_code = code2text_ds[1]["prompt"]   # a longer sample
print(f"Original ({tokenizer.count(sample_code)} tokens):")
wrap_print(sample_code[:300] + ("…" if len(sample_code) > 300 else ""))
print()

operators = [
    RemoveExtraWhitespace(),
    RemoveStopwords(),
    RemovePolitenessMarkers(),
    ContractionSubstitution(),
    DeduplicateInstructions(),
    RemovePromptHeader(),
    SentenceMerge(),
]

section("Effect of Each Operator on a Code Prompt")
for op in operators:
    result = op(sample_code)
    delta  = tokenizer.count(sample_code) - tokenizer.count(result)
    changed = result != sample_code
    status = f"−{delta} tokens" if changed else "no change"
    corrupted = ""
    if changed and delta < 0:
        corrupted = "  ← EXPANDED"
    elif changed:
        corrupted = "  ← tokens removed from code"
    print(f"  [{op.id}] {op.name:<35s} {status}{corrupted}")
    if changed:
        wrap_print(f"    After: {result[:200]}")

print()
wrap_print("Key insight: operators designed for NL prose either do nothing or")
wrap_print("corrupt the code by removing Python keywords (stopwords like 'in',")
wrap_print("'not', 'or', 'for', 'is') or identifiers that resemble common words.")

### Why `code2text` is not suitable for prompt compression

| Property | code2text | Other 5 tasks |
|-----------|-----------|---------------|
| Has a natural-language instruction | **No** — code is the only input | Yes |
| Has a few-shot block | **No** — 0-shot only | Yes (3–8 shots) |
| Prompt has separable instruction vs data | **No** — prompt = data | Yes |
| NL operators (stopwords, headers…) are safe | **No** — corrupt code tokens | Yes |
| `compress_field` has a meaningful target | **No** | Yes |

The `code2text` prompt has no instruction to compress: the code *is* the task.
Any token removed is a code token — either a keyword (`in`, `not`, `return`) or
an identifier — making the prompt semantically corrupted.
The compressor's operators were designed for natural-language instructions; applying
them to code produces either no change (most operators) or broken code (stopword removal).

**Conclusion:** `code2text` should be excluded from the compression benchmark.
`mbpp_instruct` is the right SE benchmark: it has a compressible NL instruction,
3 structured few-shot examples, and unit-test strings that can all be compressed.

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
# Note: for MMLU-Pro and BBH, LMEvalDataset cannot detect the shots boundary
# (the task renders everything into one string).  We fall back to computing
# shots ≈ full − 0-shot, which is accurate because the 0-shot field is built
# from an independent fewshot_context(doc, 0) call.

def effective_shots_mean(full_stats, zshot_stats, raw_shots_stats):
    if raw_shots_stats["mean"] > 0:
        return raw_shots_stats["mean"]
    return round(full_stats["mean"] - zshot_stats["mean"], 1)

rows = [
    ("mbpp_instruct",                                       "Code generation",       3, mbpp_full_stats, mbpp_zshot_stats, mbpp_shots_stats),
    ("mmlu_pro_computer_science",                           "CS MCQ (CoT)",          5, mmlu_full_stats, mmlu_zshot_stats, mmlu_shots_stats),
    ("gsm8k_cot",                                           "Math reasoning",        8, gsm_full_stats,  gsm_zshot_stats,  gsm_shots_stats),
    ("bbh_cot_fewshot_logical_deduction_seven_objects",     "Logical reasoning (CoT)", 3, bbh_full_stats, bbh_zshot_stats, bbh_shots_stats),
    ("leaderboard_musr_murder_mysteries",                   "Long-context reasoning", 0, musr_full_stats, musr_full_stats,  {"mean": 0}),
]

summary_rows = []
for name, domain, n_shot, full_s, zshot_s, shots_s in rows:
    shots_mean = effective_shots_mean(full_s, zshot_s, shots_s)
    summary_rows.append({
        "task": name,
        "domain": domain,
        "shots": n_shot,
        "full (words)": full_s["mean"],
        "0-shot (words)": zshot_s["mean"],
        "shots (words)": shots_mean,
        "shots %": round(shots_mean / full_s["mean"] * 100, 1) if full_s["mean"] > 0 else 0,
    })

df_summary = pd.DataFrame(summary_rows).set_index("task")
print("Cross-Task Summary (whitespace token counts)")
print(df_summary.to_string())

In [ ]:
# ── Stacked bar: 0-shot instruction vs few-shot block ─────────────────────────
task_labels = [
    "mbpp\n(3-shot)",
    "mmlu-pro\nCS (5-shot)",
    "gsm8k\n(8-shot)",
    "bbh logical\ndeduction (3-shot)",
    "MuSR\nmurder (0-shot)",
]
zshot_means = [
    mbpp_zshot_stats["mean"],
    mmlu_zshot_stats["mean"],
    gsm_zshot_stats["mean"],
    bbh_zshot_stats["mean"],
    musr_full_stats["mean"],
]
# Use full-0shot as shots when _shots detection failed (MMLU-Pro, BBH)
shots_means = [
    effective_shots_mean(mbpp_full_stats, mbpp_zshot_stats, mbpp_shots_stats),
    effective_shots_mean(mmlu_full_stats, mmlu_zshot_stats, mmlu_shots_stats),
    effective_shots_mean(gsm_full_stats,  gsm_zshot_stats,  gsm_shots_stats),
    effective_shots_mean(bbh_full_stats,  bbh_zshot_stats,  bbh_shots_stats),
    0,
]

x = np.arange(len(task_labels))
width = 0.55

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked bar
ax = axes[0]
bars1 = ax.bar(x, zshot_means, width, label="0-shot instruction", color="steelblue")
bars2 = ax.bar(x, shots_means, width, bottom=zshot_means, label="few-shot block", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(task_labels, fontsize=9)
ax.set_ylabel("Mean whitespace tokens")
ax.set_title("Prompt length breakdown: 0-shot vs few-shot block")
ax.legend()
for rect, z, s in zip(bars2, zshot_means, shots_means):
    total = z + s
    ax.text(
        rect.get_x() + rect.get_width() / 2,
        total + 5,
        f"{total:.0f}",
        ha="center", va="bottom", fontsize=8
    )

# Right: box plots of full prompt lengths
ax2 = axes[1]
all_datasets = [mbpp_ds, mmlu_ds, gsm_ds, bbh_ds, musr_ds]
all_lengths = [
    [tokenizer.count(s["prompt"]) for s in ds]
    for ds in all_datasets
]
bp = ax2.boxplot(all_lengths, labels=task_labels, patch_artist=True,
                 medianprops=dict(color="black", linewidth=2))
colors_bp = ["steelblue", "seagreen", "tomato", "mediumpurple", "goldenrod"]
for patch, color in zip(bp["boxes"], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_ylabel("Full prompt tokens (whitespace)")
ax2.set_title("Full prompt length distribution per task")
ax2.tick_params(axis="x", labelsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Compression potential heuristic: apply two simple operators and measure ───
# We apply RemoveExtraWhitespace + RemovePolitenessMarkers (the lightest operators)
# to each task's 0-shot prompts to get a lower-bound estimate of compressibility.

from prompt_piper.transformations.lexical import RemoveExtraWhitespace, RemovePolitenessMarkers
from prompt_piper.transformations.structural import DeduplicateInstructions

light_pipeline = [RemoveExtraWhitespace(), RemovePolitenessMarkers(), DeduplicateInstructions()]

def apply_pipeline(text: str) -> str:
    for op in light_pipeline:
        text = op(text)
    return text

results = []
for name, ds, field in [
    ("mbpp_instruct",                              mbpp_ds,  "prompt_0shot"),
    ("mmlu_pro_computer_science",                  mmlu_ds,  "prompt_0shot"),
    ("gsm8k_cot",                                  gsm_ds,   "prompt_0shot"),
    ("bbh_logical_deduction_seven_objects",         bbh_ds,   "prompt_0shot"),
    ("musr_murder_mysteries",                       musr_ds,  "prompt"),
]:
    originals   = [s.get(field, s["prompt"]) for s in ds]
    compressed  = [apply_pipeline(t) for t in originals]
    orig_tok    = [tokenizer.count(t) for t in originals]
    comp_tok    = [tokenizer.count(t) for t in compressed]
    ratio       = np.mean(comp_tok) / np.mean(orig_tok)
    saved       = np.mean(orig_tok) - np.mean(comp_tok)
    results.append({
        "task": name,
        "orig_mean": round(float(np.mean(orig_tok)), 1),
        "comp_mean": round(float(np.mean(comp_tok)), 1),
        "saved_mean": round(float(saved), 1),
        "ratio": round(float(ratio), 3),
    })

df_comp = pd.DataFrame(results).set_index("task")
df_comp.columns = ["orig (words)", "compressed (words)", "saved (words)", "ratio"]
print("Light compression (RemoveWhitespace + RemovePoliteness + Deduplicate) on 0-shot prompts")
print(df_comp.to_string())

---
## Key Takeaways

| Task | Total tokens (mean) | Few-shot share | Compression target |
|------|---------------------|---------------|--------------------|
| `mbpp_instruct` | ~low-medium | ~60–70% shots | Instruction + test strings + shot solutions |
| `mmlu_pro_computer_science` | ~medium-high | ~80–90% shots | CoT rationale in each of 5 shots |
| `gsm8k_cot` | ~high | ~85–90% shots | 8 long reasoning chains — high compression potential |
| `bbh_cot_logical_deduction` | ~medium-high | ~70–80% shots | Constraint narratives with repetitive phrasing |
| `musr_murder_mysteries` | ~very high | 0% (0-shot) | Entire narrative — high potential for semantic compression |

**Recommended `compress_field` setting:**
- Tasks 1–4: use `compress_field: "prompt_0shot"` to compress only the instruction, leaving shots intact — this is the most scientifically interesting regime because it isolates instruction compression from demonstration compression.
- Task 5 (MuSR): must use `compress_field: null` (full prompt), since there are no shots — all compression happens on the narrative.